# Chapter 3: Measuring Speed: Big O You Can See

The programs from Chapter 3 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Counting Steps

`count_steps.py`

In [1]:
"""Count the basic steps two functions take, as n grows."""


def find(items, target):
    """Linear search. Returns (index or -1, comparisons made)."""
    steps = 0
    for i, x in enumerate(items):
        steps += 1
        if x == target:
            return i, steps
    return -1, steps


def has_duplicate(items):
    """Compare every pair. Returns (answer, comparisons made)."""
    steps = 0
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            steps += 1
            if items[i] == items[j]:
                return True, steps
    return False, steps


print(f"{'n':>6} {'find: steps':>12} {'duplicates: steps':>18}")
for n in [10, 20, 40, 80, 160]:
    items = list(range(n))
    _, find_steps = find(items, -1)  # not there: the worst case
    _, pair_steps = has_duplicate(items)  # no duplicates: the worst case
    print(f"{n:>6} {find_steps:>12,} {pair_steps:>18,}")

     n  find: steps  duplicates: steps
    10           10                 45
    20           20                190
    40           40                780
    80           80              3,160
   160          160             12,720


## Big O Notation

`growth_table.py`

In [2]:
"""How the common growth rates compare as n grows."""

import math

print(
    f"{'n':>9} {'log2 n':>6} {'n log2 n':>10} {'n squared':>17} "
    f"{'2 to the n':>14}"
)
for n in [10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    if n <= 30:
        two_to_n = f"{2**n:,}"
    else:  # too long to print: count its digits instead
        two_to_n = f"{math.floor(n * math.log10(2)) + 1:,} digits"
    print(
        f"{n:>9,} {math.log2(n):>6.1f} {n * math.log2(n):>10,.0f} "
        f"{n * n:>17,} {two_to_n:>14}"
    )

        n log2 n   n log2 n         n squared     2 to the n
       10    3.3         33               100          1,024
      100    6.6        664            10,000      31 digits
    1,000   10.0      9,966         1,000,000     302 digits
   10,000   13.3    132,877       100,000,000   3,011 digits
  100,000   16.6  1,660,964    10,000,000,000  30,103 digits
1,000,000   19.9 19,931,569 1,000,000,000,000 301,030 digits


## Timing Code: Harder Than It Looks

`time_once.py`

In [3]:
"""Timing one call with a stopwatch: the numbers jump around."""

import time

numbers = list(range(100_000))
for attempt in range(5):
    start = time.perf_counter()
    sum(numbers)
    elapsed = time.perf_counter() - start
    print(f"attempt {attempt + 1}: {elapsed * 1e6:.0f} microseconds")

attempt 1: 1356 microseconds
attempt 2: 1542 microseconds
attempt 3: 1553 microseconds
attempt 4: 646 microseconds
attempt 5: 610 microseconds


## Reading the Ratios

`bench_growth.py`

In [4]:
"""Three functions, three growth rates, measured by doubling n."""

import bench


def first(items):  # constant: one step, whatever n is
    return items[0]


def total(items):  # linear: one step per item
    s = 0
    for x in items:
        s += x
    return s


def has_duplicate(items):  # quadratic: one step per pair of items
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i] == items[j]:
                return True
    return False


def make(n):
    return (list(range(n)),)


sizes = [250, 500, 1_000, 2_000, 4_000]
results = {
    "first item": bench.doubling(first, make, sizes),
    "sum": bench.doubling(total, make, sizes),
    "all pairs": bench.doubling(has_duplicate, make, sizes),
}
bench.table(results)
bench.chart(results, "ch03_growth.png", "Three growth rates", log=True)

         n      first item             sum       all pairs
       250         70.7 ns         3.69 us          513 us
       500      71 ns x1.0    7.46 us x2.0       2 ms x3.9
     1,000    69.7 ns x1.0    15.1 us x2.0    7.97 ms x4.0
     2,000    75.9 ns x1.1    30.8 us x2.0    38.7 ms x4.9
     4,000    73.8 ns x1.0    62.8 us x2.0     165 ms x4.3


## Best, Worst and Average Cases

`bench_cases.py`

In [5]:
"""Best, average and worst case: linear search for different targets."""

import bench

n = 100_000
items = list(range(n))
cases = {
    "best (first item)": 0,
    "middle item": n // 2,
    "worst (missing)": -1,
}
for name, target in cases.items():
    seconds = bench.measure(lambda: target in items)
    print(f"{name:>18}: {bench.show_time(seconds)}")

 best (first item): 52.8 ns


       middle item: 338 us


   worst (missing): 681 us
